# AutoGB: Parameter-Free Adaptive Granular-Ball Learning

This notebook presents **AutoGB**, a research prototype for interpretable risk modeling and selective prediction on tabular data.

## What this notebook includes

- adaptive granular-ball construction
- local risk modeling
- meta-level aggregation
- probability calibration
- ablation study
- multi-seed / multi-dataset evaluation
- risk-coverage analysis

## Suggested use

Run the notebook from top to bottom. All outputs, tables, and figures are generated automatically.


## 1. Setup and dataset preparation

This section imports dependencies, fixes seeds, defines the output directory, and prepares the benchmark datasets used in the notebook.


In [1]:
# ============================================================
# AutoGB
# GitHub Research Notebook
# ------------------------------------------------------------
# Main method = V3 (best validated branch)
# Features:
#   - 5 seeds
#   - multi-datasets
#   - mean ± std summary
#   - ablation table
#   - risk-coverage curve
#   - paper/GitHub-ready csv/json/png outputs
# ------------------------------------------------------------
# Notebook version | Google Colab | CPU-friendly
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from collections import Counter, defaultdict

from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, precision_recall_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.isotonic import IsotonicRegression

# ============================================================
# 0. Global settings
# ============================================================
SEEDS = [7, 13, 29, 42, 77]
OUT_DIR = "/content/autogb_health_v4_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

np.set_printoptions(suppress=True, precision=4)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ============================================================
# 1. Dataset preparation
# ============================================================
def get_dataset_dict():
    ds = {}

    # 1) Breast Cancer: 1=malignant(high risk), 0=benign(low risk)
    bc = load_breast_cancer()
    X_bc = bc.data
    y_bc = (bc.target == 0).astype(int)
    ds["BreastCancer"] = {
        "X": X_bc,
        "y": y_bc,
        "feature_names": list(bc.feature_names),
        "description": "Breast Cancer Wisconsin, binary risk"
    }

    # 2) Wine: class_2 -> high risk, others -> low risk
    wine = load_wine()
    X_wine = wine.data
    y_wine = (wine.target == 2).astype(int)
    ds["Wine"] = {
        "X": X_wine,
        "y": y_wine,
        "feature_names": list(wine.feature_names),
        "description": "Wine transformed to binary risk (class 2 vs rest)"
    }

    # 3) Digits: >=5 as high risk, <5 as low risk
    digits = load_digits()
    X_digits = digits.data
    y_digits = (digits.target >= 5).astype(int)
    ds["Digits"] = {
        "X": X_digits,
        "y": y_digits,
        "feature_names": [f"pixel_{i}" for i in range(X_digits.shape[1])],
        "description": "Digits transformed to binary risk (>=5 vs <5)"
    }

    return ds


## 2. Granular-ball structure

The `GranularBall` dataclass stores the statistics of each local region discovered during adaptive partitioning.


In [2]:
DATASETS = get_dataset_dict()

# ============================================================
# 2. Granular-ball structure
# ============================================================
@dataclass
class GranularBall:
    ball_id: int
    idxs: np.ndarray
    center: np.ndarray
    radius: float
    purity: float
    majority_label: int
    label_dist: dict
    density: float
    dispersion: float
    support_size: int
    parent_id: int
    level: int
    score_gain: float

# ============================================================
# 3. Main AutoGB model
# ============================================================

## 3. Main AutoGB model

This is the main framework used throughout the notebook. It combines:

1. adaptive granular-ball partitioning  
2. local risk extraction  
3. meta-level feature construction  
4. calibration for probability refinement


In [3]:
class AutoGB:
    def __init__(self, min_leaf_size=8, eps=1e-8, verbose=False, seed=42):
        self.min_leaf_size = min_leaf_size
        self.eps = eps
        self.verbose = verbose
        self.seed = seed

        self.balls = []
        self.ball_counter = 0
        self.feature_names = None
        self.X_train = None
        self.y_train = None

        self.gain_history = []
        self.level_gain_history = defaultdict(list)

        self.meta_clf = None
        self.calibrator = None
        self.meta_feature_names = None

    # -------------------------
    # Basic statistics
    # -------------------------
    def _compute_stats_dict(self, X_sub, y_sub, idxs, parent_id=-1, level=0, score_gain=0.0):
        center = X_sub.mean(axis=0)
        d = np.linalg.norm(X_sub - center, axis=1)

        radius = float(np.max(d)) if len(d) > 0 else 0.0
        dispersion = float(np.mean(d)) if len(d) > 0 else 0.0

        cnt = Counter(y_sub)
        majority_label = max(cnt.items(), key=lambda z: z[1])[0]
        purity = cnt[majority_label] / len(y_sub)

        density = len(y_sub) / (1.0 + dispersion)

        return {
            "idxs": np.array(idxs),
            "center": center,
            "radius": radius,
            "purity": float(purity),
            "majority_label": int(majority_label),
            "label_dist": dict(cnt),
            "density": float(density),
            "dispersion": float(dispersion),
            "support_size": int(len(idxs)),
            "parent_id": int(parent_id),
            "level": int(level),
            "score_gain": float(score_gain)
        }

    def _build_ball_obj(self, stats):
        ball = GranularBall(
            ball_id=self.ball_counter,
            idxs=stats["idxs"],
            center=stats["center"],
            radius=stats["radius"],
            purity=stats["purity"],
            majority_label=stats["majority_label"],
            label_dist=stats["label_dist"],
            density=stats["density"],
            dispersion=stats["dispersion"],
            support_size=stats["support_size"],
            parent_id=stats["parent_id"],
            level=stats["level"],
            score_gain=stats["score_gain"]
        )
        self.ball_counter += 1
        return ball

    # -------------------------
    # Geometric splitting
    # -------------------------
    def _split_ball(self, X_sub, y_sub, idxs):
        n = len(X_sub)
        if n < 2 * self.min_leaf_size:
            return None

        center = X_sub.mean(axis=0)
        d0 = np.linalg.norm(X_sub - center, axis=1)
        a_idx = np.argmax(d0)
        a = X_sub[a_idx]

        da = np.linalg.norm(X_sub - a, axis=1)
        b_idx = np.argmax(da)
        b = X_sub[b_idx]

        dist_a = np.linalg.norm(X_sub - a, axis=1)
        dist_b = np.linalg.norm(X_sub - b, axis=1)

        left_mask = dist_a <= dist_b
        right_mask = ~left_mask

        if left_mask.sum() < self.min_leaf_size or right_mask.sum() < self.min_leaf_size:
            return None

        return (
            X_sub[left_mask], y_sub[left_mask], np.array(idxs)[left_mask],
            X_sub[right_mask], y_sub[right_mask], np.array(idxs)[right_mask]
        )

    # -------------------------
    # Split gain
    # -------------------------
    def _split_gain(self, parent_stats, left_stats, right_stats):
        n_parent = parent_stats["support_size"]
        n_left = left_stats["support_size"]
        n_right = right_stats["support_size"]

        child_purity = (n_left * left_stats["purity"] + n_right * right_stats["purity"]) / n_parent
        purity_gain = child_purity - parent_stats["purity"]

        child_radius = (n_left * left_stats["radius"] + n_right * right_stats["radius"]) / n_parent
        radius_gain = (parent_stats["radius"] - child_radius) / (parent_stats["radius"] + self.eps)

        child_disp = (n_left * left_stats["dispersion"] + n_right * right_stats["dispersion"]) / n_parent
        disp_gain = (parent_stats["dispersion"] - child_disp) / (parent_stats["dispersion"] + self.eps)

        balance = min(n_left, n_right) / max(n_left, n_right)

        gain = np.mean([
            1.25 * purity_gain,
            radius_gain,
            disp_gain,
            balance
        ])

        detail = {
            "purity_gain": float(purity_gain),
            "radius_gain": float(radius_gain),
            "disp_gain": float(disp_gain),
            "balance": float(balance),
            "gain": float(gain)
        }
        return gain, detail

    def _should_split(self, parent_stats, left_stats, right_stats, level):
        gain, detail = self._split_gain(parent_stats, left_stats, right_stats)

        if detail["purity_gain"] <= 0 and detail["radius_gain"] <= 0 and detail["disp_gain"] <= 0:
            return False, gain, detail

        highly_mixed_parent = parent_stats["purity"] < 0.82
        if highly_mixed_parent and min(left_stats["support_size"], right_stats["support_size"]) >= self.min_leaf_size:
            if gain > -0.02:
                return True, gain, detail

        past = self.level_gain_history[level] if len(self.level_gain_history[level]) > 0 else self.gain_history

        if len(past) < 6:
            positive_count = sum([
                detail["purity_gain"] > 0,
                detail["radius_gain"] > 0,
                detail["disp_gain"] > 0
            ])
            ok = (gain > 0) and (positive_count >= 2)
            return ok, gain, detail

        mu = np.mean(past)
        sd = np.std(past) + self.eps
        z = (gain - mu) / sd
        detail["gain_z"] = float(z)

        if parent_stats["purity"] < 0.90:
            ok = (gain > -0.01) and (z >= -0.60)
        else:
            ok = (gain > 0) and (z >= -0.35)

        return ok, gain, detail

    # -------------------------
    # Recursive growth
    # -------------------------
    def _grow(self, X_sub, y_sub, idxs, parent_id=-1, level=0):
        parent_stats = self._compute_stats_dict(
            X_sub, y_sub, idxs, parent_id=parent_id, level=level, score_gain=0.0
        )

        split_res = self._split_ball(X_sub, y_sub, idxs)
        if split_res is None:
            leaf_ball = self._build_ball_obj(parent_stats)
            self.balls.append(leaf_ball)
            return

        X_l, y_l, idx_l, X_r, y_r, idx_r = split_res

        left_stats = self._compute_stats_dict(X_l, y_l, idx_l, parent_id=-1, level=level + 1)
        right_stats = self._compute_stats_dict(X_r, y_r, idx_r, parent_id=-1, level=level + 1)

        should_split, gain, detail = self._should_split(parent_stats, left_stats, right_stats, level)

        if not should_split:
            parent_stats["score_gain"] = gain
            leaf_ball = self._build_ball_obj(parent_stats)
            self.balls.append(leaf_ball)
            return

        self.gain_history.append(gain)
        self.level_gain_history[level].append(gain)

        self._grow(X_l, y_l, idx_l, parent_id=parent_id, level=level + 1)
        self._grow(X_r, y_r, idx_r, parent_id=parent_id, level=level + 1)

    # -------------------------
    # Ball weights
    # -------------------------
    def _ball_weight(self, x, ball):
        dist = np.linalg.norm(x - ball.center)
        norm_dist = dist / (ball.radius + 1.0)

        risk_frac = ball.label_dist.get(1, 0) / max(sum(ball.label_dist.values()), 1)

        purity_term = (0.35 + 0.65 * ball.purity)
        locality_term = 1.0 / (1.0 + norm_dist)
        density_term = np.log1p(ball.density)
        radius_penalty = 1.0 / (1.0 + 0.12 * ball.radius)
        ambiguity_penalty = 1.0 - 0.35 * (1.0 - ball.purity)

        weight = locality_term * purity_term * density_term * radius_penalty * ambiguity_penalty

        return {
            "ball_id": ball.ball_id,
            "level": ball.level,
            "dist": float(dist),
            "norm_dist": float(norm_dist),
            "weight": float(weight),
            "purity": float(ball.purity),
            "density": float(ball.density),
            "risk_frac": float(risk_frac),
            "majority_label": int(ball.majority_label),
            "ball_size": int(ball.support_size),
            "radius": float(ball.radius),
            "dispersion": float(ball.dispersion),
            "center": ball.center
        }

    # -------------------------
    # Support balls
    # -------------------------
    def _select_support_balls(self, x):
        rows = [self._ball_weight(x, b) for b in self.balls]
        df = pd.DataFrame(rows).sort_values("weight", ascending=False).reset_index(drop=True)

        total_w = df["weight"].sum() + self.eps
        cumsum_ratio = (df["weight"].cumsum() / total_w).values

        keep_idx = np.where(cumsum_ratio <= 0.85)[0]
        if len(keep_idx) == 0:
            m = 2 if len(df) >= 2 else 1
        else:
            m = int(keep_idx[-1]) + 2

        m = min(max(m, 2), len(df))
        return df.iloc[:m].copy().reset_index(drop=True)

    # -------------------------
    # Meta features
    # -------------------------
    def _extract_meta_features_one(self, x):
        support = self._select_support_balls(x)

        w = support["weight"].values
        w_norm = w / (w.sum() + self.eps)

        risk = support["risk_frac"].values
        purity = support["purity"].values
        radius = support["radius"].values
        density = support["density"].values
        dist = support["dist"].values
        level = support["level"].values
        size = support["ball_size"].values
        majority = support["majority_label"].values
        dispersion = support["dispersion"].values

        weighted_risk = np.sum(w_norm * risk)
        weighted_purity = np.sum(w_norm * purity)
        weighted_radius = np.sum(w_norm * radius)
        weighted_density = np.sum(w_norm * density)
        weighted_dist = np.sum(w_norm * dist)
        weighted_level = np.sum(w_norm * level)
        weighted_size = np.sum(w_norm * size)
        weighted_disp = np.sum(w_norm * dispersion)

        risk_var = np.sum(w_norm * (risk - weighted_risk) ** 2)
        majority_conflict = np.sum(w_norm * (majority != int(weighted_risk >= 0.5)).astype(float))
        top1_margin = float(support.loc[0, "weight"] - support.loc[1, "weight"]) if len(support) >= 2 else float(support.loc[0, "weight"])
        min_dist = float(dist.min())
        top1_purity = float(support.loc[0, "purity"])
        top1_risk = float(support.loc[0, "risk_frac"])
        support_count = float(len(support))

        features = np.array([
            weighted_risk,
            weighted_purity,
            weighted_radius,
            weighted_density,
            weighted_dist,
            weighted_level,
            weighted_size,
            weighted_disp,
            risk_var,
            majority_conflict,
            top1_margin,
            min_dist,
            top1_purity,
            top1_risk,
            support_count,
            np.max(risk),
            np.min(risk),
            np.mean(risk),
            np.std(risk),
            np.mean(purity),
            np.std(purity),
            np.mean(radius),
            np.std(radius),
            np.sum(w),
        ], dtype=float)

        uncertainty = float(0.55 * risk_var + 0.25 * majority_conflict + 0.20 * (1.0 - top1_purity))

        return features, support, uncertainty

    def _extract_meta_features(self, X):
        feats = []
        uncerts = []
        for x in X:
            f, _, u = self._extract_meta_features_one(x)
            feats.append(f)
            uncerts.append(u)
        return np.vstack(feats), np.array(uncerts)

    # -------------------------
    # Fit
    # -------------------------
    def fit(self, X_train, y_train, X_val=None, y_val=None, feature_names=None):
        self.X_train = X_train.copy()
        self.y_train = y_train.copy()
        self.feature_names = feature_names if feature_names is not None else [f"f{i}" for i in range(X_train.shape[1])]

        self.balls = []
        self.ball_counter = 0
        self.gain_history = []
        self.level_gain_history = defaultdict(list)

        idxs = np.arange(len(X_train))
        self._grow(X_train, y_train, idxs, parent_id=-1, level=0)

        Z_train, U_train = self._extract_meta_features(X_train)
        self.meta_feature_names = [
            "weighted_risk", "weighted_purity", "weighted_radius", "weighted_density",
            "weighted_dist", "weighted_level", "weighted_size", "weighted_disp",
            "risk_var", "majority_conflict", "top1_margin", "min_dist",
            "top1_purity", "top1_risk", "support_count", "risk_max", "risk_min",
            "risk_mean", "risk_std", "purity_mean", "purity_std",
            "radius_mean", "radius_std", "weight_sum"
        ]

        self.meta_clf = LogisticRegression(
            C=1.0,
            max_iter=4000,
            class_weight="balanced",
            random_state=self.seed
        )
        self.meta_clf.fit(Z_train, y_train)

        if X_val is not None and y_val is not None:
            Z_val, _ = self._extract_meta_features(X_val)
            raw_val_prob = self.meta_clf.predict_proba(Z_val)[:, 1]
            self.calibrator = IsotonicRegression(out_of_bounds="clip")
            self.calibrator.fit(raw_val_prob, y_val)

        return self

    def predict_proba_raw(self, X):
        Z, _ = self._extract_meta_features(X)
        return self.meta_clf.predict_proba(Z)

    def predict_proba(self, X):
        raw_prob = self.predict_proba_raw(X)[:, 1]
        if self.calibrator is not None:
            p = self.calibrator.predict(raw_prob)
        else:
            p = raw_prob
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.column_stack([1 - p, p])

    def predict(self, X, threshold=0.5):
        p = self.predict_proba(X)[:, 1]
        return (p >= threshold).astype(int)

    def uncertainty(self, X):
        _, U = self._extract_meta_features(X)
        return U


## 4. Ablation models

This section defines three variants used for ablation:

- **AutoGBDirect**: direct granular-ball risk aggregation  
- **AutoGBMetaNoCal**: granular balls + meta learner, without calibration  
- **AutoGB**: full model with calibration


In [4]:
# 4. Ablation models
# ============================================================
class AutoGBDirect:
    """
    Ablation A:
    Granular-ball structure only + direct risk-weighted averaging
    No meta learner and no calibration
    """
    def __init__(self, min_leaf_size=8, eps=1e-8, verbose=False, seed=42):
        self.min_leaf_size = min_leaf_size
        self.eps = eps
        self.verbose = verbose
        self.seed = seed
        self.base = AutoGB(min_leaf_size=min_leaf_size, eps=eps, verbose=verbose, seed=seed)

    def fit(self, X_train, y_train, feature_names=None):
        # Train the structural layer only; do not train the meta learner
        self.base.X_train = X_train.copy()
        self.base.y_train = y_train.copy()
        self.base.feature_names = feature_names if feature_names is not None else [f"f{i}" for i in range(X_train.shape[1])]
        self.base.balls = []
        self.base.ball_counter = 0
        self.base.gain_history = []
        self.base.level_gain_history = defaultdict(list)
        idxs = np.arange(len(X_train))
        self.base._grow(X_train, y_train, idxs, parent_id=-1, level=0)
        return self

    def predict_proba(self, X):
        probs = []
        for x in X:
            support = self.base._select_support_balls(x)
            w = support["weight"].values
            r = support["risk_frac"].values
            if np.sum(w) <= self.base.eps:
                p = np.mean(self.base.y_train)
            else:
                p = np.sum(w * r) / (np.sum(w) + self.base.eps)
            p = np.clip(p, 1e-6, 1 - 1e-6)
            probs.append([1 - p, p])
        return np.array(probs)

    def predict(self, X, threshold=0.5):
        p = self.predict_proba(X)[:, 1]
        return (p >= threshold).astype(int)

    def uncertainty(self, X):
        # Use support-risk variance directly
        U = []
        for x in X:
            support = self.base._select_support_balls(x)
            w = support["weight"].values
            risk = support["risk_frac"].values
            w_norm = w / (w.sum() + self.base.eps)
            wr = np.sum(w_norm * risk)
            risk_var = np.sum(w_norm * (risk - wr) ** 2)
            U.append(float(risk_var))
        return np.array(U)

class AutoGBMetaNoCal:
    """
    Ablation B:
    Granular-ball structure + meta learner
    No calibration
    """
    def __init__(self, min_leaf_size=8, eps=1e-8, verbose=False, seed=42):
        self.model = AutoGB(min_leaf_size=min_leaf_size, eps=eps, verbose=verbose, seed=seed)

    def fit(self, X_train, y_train, feature_names=None):
        self.model.fit(X_train, y_train, X_val=None, y_val=None, feature_names=feature_names)
        self.model.calibrator = None
        return self

    def predict_proba(self, X):
        return self.model.predict_proba_raw(X)

    def predict(self, X, threshold=0.5):
        p = self.predict_proba(X)[:, 1]
        return (p >= threshold).astype(int)

    def uncertainty(self, X):
        return self.model.uncertainty(X)

class AutoGBMetaCal:
    """
    Ablation C:
    Granular-ball structure + meta learner + calibration
    """
    def __init__(self, min_leaf_size=8, eps=1e-8, verbose=False, seed=42):
        self.model = AutoGB(min_leaf_size=min_leaf_size, eps=eps, verbose=verbose, seed=seed)

    def fit(self, X_train, y_train, X_val, y_val, feature_names=None):
        self.model.fit(X_train, y_train, X_val=X_val, y_val=y_val, feature_names=feature_names)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def predict(self, X, threshold=0.5):
        p = self.predict_proba(X)[:, 1]
        return (p >= threshold).astype(int)

    def uncertainty(self, X):
        return self.model.uncertainty(X)


## 5. Utility functions

These helper functions compute metrics, choose thresholds, build risk-coverage curves, and save figures.


In [5]:
# 5. Utility functions
# ============================================================
def compute_metrics(y_true, prob, pred):
    out = {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "Brier": brier_score_loss(y_true, prob)
    }
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, prob)
    out["PR_AUC"] = auc(recall_curve, precision_curve)
    return out

def format_mean_std(x):
    x = np.array(x, dtype=float)
    return f"{x.mean():.4f} ± {x.std(ddof=0):.4f}"

def choose_best_threshold_by_f1(y_val, prob_val):
    thresholds = np.linspace(0.10, 0.90, 161)
    best_thr = 0.5
    best_f1 = -1
    for thr in thresholds:
        pred = (prob_val >= thr).astype(int)
        f1 = f1_score(y_val, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return float(best_thr), float(best_f1)

def risk_coverage_curve(y_true, prob, unc, threshold, fracs=np.linspace(0.30, 1.0, 15)):
    order = np.argsort(unc)
    rows = []
    for frac in fracs:
        k = max(1, int(len(order) * frac))
        keep = order[:k]
        y_k = y_true[keep]
        p_k = prob[keep]
        pred_k = (p_k >= threshold).astype(int)
        rows.append({
            "coverage": float(frac),
            "F1": f1_score(y_k, pred_k, zero_division=0),
            "Recall": recall_score(y_k, pred_k, zero_division=0),
            "Accuracy": accuracy_score(y_k, pred_k),
        })
    return pd.DataFrame(rows)

def save_risk_coverage_plot(df_curve, title, path):
    plt.figure(figsize=(7, 5))
    plt.plot(df_curve["coverage"], df_curve["F1"], marker="o", label="F1")
    plt.plot(df_curve["coverage"], df_curve["Recall"], marker="s", label="Recall")
    plt.xlabel("Coverage")
    plt.ylabel("Metric")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


## 6. Full experiment pipeline

This section runs the full benchmark across multiple seeds and datasets, then exports summary tables and plots.


In [6]:
def run_one_dataset_one_seed(ds_name, ds_obj, seed, verbose=False):
    X = ds_obj["X"]
    y = ds_obj["y"]
    feature_names = ds_obj["feature_names"]

    # Split data
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=seed
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=seed
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    # --------------------------------------------------------
    # Baselines
    # --------------------------------------------------------
    baselines = {
        "LogisticRegression": LogisticRegression(max_iter=3000, random_state=seed),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=seed),
        "SVM-RBF": SVC(probability=True, kernel="rbf", C=2.0, gamma="scale", random_state=seed),
        "KNN": KNeighborsClassifier(n_neighbors=9),
    }

    result_rows = []

    for name, model in baselines.items():
        model.fit(X_train_s, y_train)
        prob_test = model.predict_proba(X_test_s)[:, 1]
        pred_test = (prob_test >= 0.5).astype(int)
        metrics = compute_metrics(y_test, prob_test, pred_test)
        result_rows.append({
            "Dataset": ds_name,
            "Seed": seed,
            "Model": name,
            **metrics,
            "Threshold": 0.5
        })

    # --------------------------------------------------------
    # Ablation A: Direct
    # --------------------------------------------------------
    gb_direct = AutoGBDirect(min_leaf_size=8, verbose=verbose, seed=seed)
    gb_direct.fit(X_train_s, y_train, feature_names=feature_names)
    prob_val_direct = gb_direct.predict_proba(X_val_s)[:, 1]
    thr_direct, _ = choose_best_threshold_by_f1(y_val, prob_val_direct)
    prob_test_direct = gb_direct.predict_proba(X_test_s)[:, 1]
    pred_test_direct = (prob_test_direct >= thr_direct).astype(int)
    metrics_direct = compute_metrics(y_test, prob_test_direct, pred_test_direct)
    result_rows.append({
        "Dataset": ds_name,
        "Seed": seed,
        "Model": "AutoGB-Direct",
        **metrics_direct,
        "Threshold": thr_direct
    })

    # --------------------------------------------------------
    # Ablation B: Meta no calibration
    # --------------------------------------------------------
    gb_meta_nocal = AutoGBMetaNoCal(min_leaf_size=8, verbose=verbose, seed=seed)
    gb_meta_nocal.fit(X_train_s, y_train, feature_names=feature_names)
    prob_val_meta_nocal = gb_meta_nocal.predict_proba(X_val_s)[:, 1]
    thr_meta_nocal, _ = choose_best_threshold_by_f1(y_val, prob_val_meta_nocal)
    prob_test_meta_nocal = gb_meta_nocal.predict_proba(X_test_s)[:, 1]
    pred_test_meta_nocal = (prob_test_meta_nocal >= thr_meta_nocal).astype(int)
    metrics_meta_nocal = compute_metrics(y_test, prob_test_meta_nocal, pred_test_meta_nocal)
    result_rows.append({
        "Dataset": ds_name,
        "Seed": seed,
        "Model": "AutoGB-Meta-NoCal",
        **metrics_meta_nocal,
        "Threshold": thr_meta_nocal
    })

    # --------------------------------------------------------
    # Ablation C / Main: Meta + calibration
    # --------------------------------------------------------
    gb_meta_cal = AutoGBMetaCal(min_leaf_size=8, verbose=verbose, seed=seed)
    gb_meta_cal.fit(X_train_s, y_train, X_val_s, y_val, feature_names=feature_names)
    prob_val_meta_cal = gb_meta_cal.predict_proba(X_val_s)[:, 1]
    thr_meta_cal, _ = choose_best_threshold_by_f1(y_val, prob_val_meta_cal)
    prob_test_meta_cal = gb_meta_cal.predict_proba(X_test_s)[:, 1]
    pred_test_meta_cal = (prob_test_meta_cal >= thr_meta_cal).astype(int)
    metrics_meta_cal = compute_metrics(y_test, prob_test_meta_cal, pred_test_meta_cal)
    result_rows.append({
        "Dataset": ds_name,
        "Seed": seed,
        "Model": "AutoGB",
        **metrics_meta_cal,
        "Threshold": thr_meta_cal
    })

    # Risk-coverage curve: main method
    unc_test = gb_meta_cal.uncertainty(X_test_s)
    df_curve = risk_coverage_curve(y_test, prob_test_meta_cal, unc_test, thr_meta_cal)
    df_curve["Dataset"] = ds_name
    df_curve["Seed"] = seed
    df_curve["Model"] = "AutoGB"

    # Additional structural information
    n_balls = len(gb_meta_cal.model.balls)
    avg_purity = np.mean([b.purity for b in gb_meta_cal.model.balls]) if n_balls > 0 else np.nan
    avg_radius = np.mean([b.radius for b in gb_meta_cal.model.balls]) if n_balls > 0 else np.nan

    structure_info = {
        "Dataset": ds_name,
        "Seed": seed,
        "Model": "AutoGB",
        "n_balls": n_balls,
        "avg_purity": float(avg_purity),
        "avg_radius": float(avg_radius),
        "threshold": float(thr_meta_cal)
    }

    return pd.DataFrame(result_rows), df_curve, structure_info

# ============================================================
# 7. Full evaluation
# ============================================================
all_results = []
all_curves = []
all_structure = []

print("=" * 96)
print("AutoGB | multi-seed + multi-dataset + ablation evaluation started")
print("=" * 96)

for ds_name, ds_obj in DATASETS.items():
    print(f"\n======================== DATASET: {ds_name} ========================")
    for seed in SEEDS:
        print(f"[seed={seed}] running ...")
        df_res, df_curve, info = run_one_dataset_one_seed(ds_name, ds_obj, seed, verbose=False)
        all_results.append(df_res)
        all_curves.append(df_curve)
        all_structure.append(info)

df_all_results = pd.concat(all_results, axis=0, ignore_index=True)
df_all_curves = pd.concat(all_curves, axis=0, ignore_index=True)
df_structure = pd.DataFrame(all_structure)

print("\nEvaluation completed.")
print()

# ============================================================
# 8. Mean ± std summary
# ============================================================
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "Brier", "PR_AUC"]

summary_rows = []
for (ds_name, model_name), g in df_all_results.groupby(["Dataset", "Model"]):
    row = {"Dataset": ds_name, "Model": model_name}
    for m in metric_cols:
        row[f"{m}_mean"] = g[m].mean()
        row[f"{m}_std"] = g[m].std(ddof=0)
        row[m] = format_mean_std(g[m].values)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# Sort each dataset by descending F1_mean
df_summary = df_summary.sort_values(["Dataset", "F1_mean", "ROC_AUC_mean"], ascending=[True, False, False]).reset_index(drop=True)

print("=" * 96)
print("Mean ± std summary by dataset")
print("=" * 96)
display_cols = ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "Brier", "PR_AUC"]
print(df_summary[display_cols].to_string(index=False))
print()

# ============================================================
# 9. Ablation table
# ============================================================
ablation_models = ["AutoGB-Direct", "AutoGB-Meta-NoCal", "AutoGB"]

ablation_rows = []
for ds_name in DATASETS.keys():
    tmp = df_summary[(df_summary["Dataset"] == ds_name) & (df_summary["Model"].isin(ablation_models))].copy()
    tmp = tmp.sort_values("F1_mean", ascending=False)
    ablation_rows.append(tmp)

df_ablation = pd.concat(ablation_rows, axis=0, ignore_index=True)

print("=" * 96)
print("Ablation table (main method pipeline)")
print("=" * 96)
print(df_ablation[display_cols].to_string(index=False))
print()

# ============================================================
# 10. Risk-coverage summary (main method averaged across seeds)
# ============================================================
df_curve_mean = (
    df_all_curves.groupby(["Dataset", "coverage", "Model"], as_index=False)[["F1", "Recall", "Accuracy"]]
    .mean()
)

for ds_name in DATASETS.keys():
    tmp = df_curve_mean[(df_curve_mean["Dataset"] == ds_name) & (df_curve_mean["Model"] == "AutoGB")].copy()
    plot_path = os.path.join(OUT_DIR, f"risk_coverage_curve_{ds_name}.png")
    save_risk_coverage_plot(tmp, f"Risk-Coverage Curve | {ds_name} | AutoGB-V4", plot_path)

# ============================================================
# 11. Additional plot: main method vs baselines by dataset (mean F1)
# ============================================================
for ds_name in DATASETS.keys():
    tmp = df_summary[df_summary["Dataset"] == ds_name].copy()
    tmp = tmp.sort_values("F1_mean", ascending=False)

    plt.figure(figsize=(8, 5))
    plt.bar(tmp["Model"], tmp["F1_mean"])
    plt.xticks(rotation=20)
    plt.ylabel("F1 mean")
    plt.title(f"F1 Comparison | {ds_name} | 5 seeds")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"f1_comparison_{ds_name}.png"), dpi=180)
    plt.close()

# ============================================================
# 12. Appendix-style wide table
# ============================================================
appendix_rows = []
for ds_name in DATASETS.keys():
    tmp = df_summary[df_summary["Dataset"] == ds_name].copy()
    for _, r in tmp.iterrows():
        appendix_rows.append({
            "Dataset": ds_name,
            "Model": r["Model"],
            "Accuracy": r["Accuracy"],
            "Precision": r["Precision"],
            "Recall": r["Recall"],
            "F1": r["F1"],
            "ROC_AUC": r["ROC_AUC"],
            "Brier": r["Brier"],
            "PR_AUC": r["PR_AUC"],
        })
df_appendix = pd.DataFrame(appendix_rows)

# ============================================================
# 13. Save outputs
# ============================================================
df_all_results.to_csv(os.path.join(OUT_DIR, "all_seed_results.csv"), index=False)
df_summary.to_csv(os.path.join(OUT_DIR, "summary_mean_std.csv"), index=False)
df_ablation.to_csv(os.path.join(OUT_DIR, "ablation_table.csv"), index=False)
df_all_curves.to_csv(os.path.join(OUT_DIR, "risk_coverage_all_seeds.csv"), index=False)
df_curve_mean.to_csv(os.path.join(OUT_DIR, "risk_coverage_mean.csv"), index=False)
df_structure.to_csv(os.path.join(OUT_DIR, "structure_stats.csv"), index=False)
df_appendix.to_csv(os.path.join(OUT_DIR, "appendix_table.csv"), index=False)

summary_json = {
    "project": "AutoGB",
    "seeds": SEEDS,
    "datasets": {
        k: {
            "n_samples": int(v["X"].shape[0]),
            "n_features": int(v["X"].shape[1]),
            "positive_ratio": float(v["y"].mean()),
            "description": v["description"]
        } for k, v in DATASETS.items()
    },
    "main_model": "AutoGB",
    "ablations": ablation_models,
    "baselines": ["LogisticRegression", "RandomForest", "SVM-RBF", "KNN"],
}

with open(os.path.join(OUT_DIR, "run_config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

# ============================================================
# 14. Console summary
# ============================================================
print("=" * 96)
print(f"Results exported to: {OUT_DIR}")
print("Saved files:")
for fn in sorted(os.listdir(OUT_DIR)):
    print(" -", fn)
print("=" * 96)

print()
print("Suggested takeaways:")
print("1) The main method is AutoGB (granular balls + meta learner + calibration).")
print("2) The ablation table shows the progression from Direct -> Meta -> Meta+Calibration.")
print("3) All results are based on 5 seeds and reported as mean ± std, which is suitable for papers and GitHub.")
print("4) The current multi-dataset setup is a methodology-oriented evaluation scaffold and can be replaced with more formal tabular benchmarks.")
print("5) Start with summary_mean_std.csv, ablation_table.csv, and appendix_table.csv.")

AutoGB | multi-seed + multi-dataset + ablation evaluation started

======================== DATASET: BreastCancer ========================
[seed=7] running ...
[seed=13] running ...
[seed=29] running ...
[seed=42] running ...
[seed=77] running ...

======================== DATASET: Wine ========================
[seed=7] running ...
[seed=13] running ...
[seed=29] running ...
[seed=42] running ...
[seed=77] running ...

======================== DATASET: Digits ========================
[seed=7] running ...
[seed=13] running ...
[seed=29] running ...
[seed=42] running ...
[seed=77] running ...

Evaluation completed.

Mean ± std summary by dataset
     Dataset              Model        Accuracy       Precision          Recall              F1         ROC_AUC           Brier          PR_AUC
BreastCancer            SVM-RBF 0.9842 ± 0.0066 0.9856 ± 0.0118 0.9714 ± 0.0095 0.9784 ± 0.0090 0.9973 ± 0.0020 0.0179 ± 0.0051 0.9961 ± 0.0026
BreastCancer LogisticRegression 0.9754 ± 0.0102 0.9952 ± 0.0